In [1]:
# Setup: imports and environment checks
import os
import sys
from pathlib import Path

# Ensure project paths
ROOT = Path(r"c:\Users\junhongs\Desktop\capstone\evaluation")
MATERIAL_DIR = ROOT / "material"
OUTPUT_DIR = ROOT / "dataset" / "uas_dataset"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Find all PDF files in material directory
pdf_files = list(MATERIAL_DIR.glob("*.pdf"))
print(f"Found {len(pdf_files)} PDF files in material directory:")
for pdf in pdf_files:
    size_kb = pdf.stat().st_size / 1024
    print(f"  - {pdf.name} ({size_kb:.1f} KB)")

# Soft dependency checks
missing = []
for pkg in ["ragas", "langchain_community", "langchain_openai", "openai", "pypdf", "pandas"]:
    try:
        __import__(pkg)
    except Exception:
        missing.append(pkg)

if missing:
    print("Missing packages detected:\n - " + "\n - ".join(missing))
    print("Install them in this kernel, for example:")
    print("%pip install ragas langchain-community langchain-openai openai pypdf pandas tqdm")
else:
    print("All required packages found.")

Found 1 PDF files in material directory:
  - USO-Administration-Guide-6.0.1-GA (AI).pdf (10199.5 KB)


c:\Users\junhongs\Desktop\capstone\evaluation\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


All required packages found.


In [2]:
# Shared utility functions
import numpy as np
import pandas as pd

def _is_nonempty_value(x):
    """Check if a value is non-empty (handles strings, lists, arrays, NaN)"""
    if x is None:
        return False
    if isinstance(x, str):
        return x.strip() != ""
    if isinstance(x, (list, tuple, set, dict)):
        return len(x) > 0
    if isinstance(x, np.ndarray):
        return x.size > 0
    try:
        import math
        if isinstance(x, float) and math.isnan(x):
            return False
    except Exception:
        pass
    return True

def infer_columns(row):
    """Extract query, ground_truth, and contexts from a RAGAS DataFrame row"""
    # Query
    query = None
    for k in ["user_input", "question", "query", "prompt"]:
        if k in row and pd.notna(row[k]):
            query = row[k]
            break
    
    # Ground truth
    gt = None
    for k in ["reference", "ground_truth", "expected_output", "answer"]:
        if k in row:
            val = row[k]
            if _is_nonempty_value(val):
                gt = val
                break
    
    # Contexts
    contexts = None
    for k in ["contexts", "reference_contexts", "contexts_text", "documents"]:
        if k in row:
            val = row[k]
            if not _is_nonempty_value(val):
                continue
            # Ensure list[str]
            if isinstance(val, str):
                contexts = [val]
            elif isinstance(val, (list, tuple)):
                first = val[0] if len(val) else None
                if isinstance(first, dict) and "page_content" in first:
                    contexts = [d.get("page_content", "") for d in val]
                else:
                    contexts = [str(v) for v in val]
            elif isinstance(val, np.ndarray):
                if val.size: 
                    first = val.flat[0]
                    if isinstance(first, dict) and "page_content" in first:
                        contexts = [d.get("page_content", "") for d in val.tolist()]
                    else:
                        contexts = [str(v) for v in val.tolist()]
            elif isinstance(val, dict) and "page_content" in val:
                contexts = [val.get("page_content", "")]            
            else:
                contexts = [str(val)]
            break
    
    return query, gt, contexts or []

In [3]:
# Process all PDFs in material directory
from langchain_community.document_loaders import PyPDFLoader
# Requires: %pip install langchain-experimental
from langchain_experimental.text_splitter import SemanticChunker
from langchain_openai import OpenAIEmbeddings

# Storage for all processed documents
all_pdf_data = []

for pdf_path in pdf_files:
    print(f"\n{'='*60}")
    print(f"Processing: {pdf_path.name}")
    print(f"{'='*60}")
    
    # Determine question count based on file size
    # User requested 100 questions for large files (like 10MB)
    size_kb = pdf_path.stat().st_size / 1024
    if size_kb > 1000: # If larger than 1MB
        testset_size = 100
    else:
        testset_size = 20
    
    # Load PDF
    loader = PyPDFLoader(str(pdf_path))
    docs = loader.load()
    total_chars = sum(len(d.page_content) for d in docs)
    
    print(f" Pages: {len(docs)} | Characters: {total_chars:,}")
    print(f" Will generate: {testset_size} questions (size: {size_kb:.1f} KB)")
    
    # Check content quality
    if total_chars < 1000:
        print(f" WARNING: Very little text extracted! Skipping this PDF.")
        continue
    
    # Chunk the document using SEMANTIC CHUNKING
    # This groups text by semantic similarity rather than fixed size.
    # It calculates embeddings for sentences and splits when the meaning changes.
    print(" Splitting documents semantically...")
    embeddings = OpenAIEmbeddings(model='text-embedding-3-small')
    text_splitter = SemanticChunker(
        embeddings,
        breakpoint_threshold_type="percentile" # Splits when semantic difference is high
    )
    chunked_docs = text_splitter.split_documents(docs)
    
    # FILTER: Remove tiny chunks that are too small for question generation
    # Semantic chunking can sometimes produce very small fragments (headers, footers).
    # RAGAS struggles with these, so we filter them out.
    original_count = len(chunked_docs)
    chunked_docs = [d for d in chunked_docs if len(d.page_content) > 50]
    filtered_count = original_count - len(chunked_docs)
    
    if filtered_count > 0:
        print(f" Filtered out {filtered_count} tiny chunks (< 50 chars)")
    
    # Calculate stats
    chunk_sizes = [len(d.page_content) for d in chunked_docs]
    avg_size = sum(chunk_sizes)//len(chunk_sizes) if chunk_sizes else 0
    print(f"Chunks: {len(chunked_docs)} | Avg size: {avg_size} chars")
    
    # Store for later processing
    all_pdf_data.append({
        'pdf_path': pdf_path,
        'pdf_name': pdf_path.stem,
        'chunks': chunked_docs,
        'testset_size': testset_size,
        'total_chars': total_chars
    })

print(f"\n Successfully loaded {len(all_pdf_data)} PDFs for processing")


Processing: USO-Administration-Guide-6.0.1-GA (AI).pdf
 Pages: 309 | Characters: 366,517
 Will generate: 100 questions (size: 10199.5 KB)
 Splitting documents semantically...
 Pages: 309 | Characters: 366,517
 Will generate: 100 questions (size: 10199.5 KB)
 Splitting documents semantically...
 Filtered out 22 tiny chunks (< 50 chars)
Chunks: 558 | Avg size: 655 chars

 Successfully loaded 1 PDFs for processing
 Filtered out 22 tiny chunks (< 50 chars)
Chunks: 558 | Avg size: 655 chars

 Successfully loaded 1 PDFs for processing


In [4]:
# Verify API key is visible to this kernel and optionally load from .env
import os

# Optional: auto-load from a .env file if present
try:
    from dotenv import load_dotenv  # type: ignore
    loaded = load_dotenv()
    if loaded:
        print("Loaded environment from .env")
except Exception:
    pass  # python-dotenv not installed; that's okay

api_key = os.environ.get("OPENAI_API_KEY", "")
masked = (api_key[:4] + "***" + api_key[-4:]) if api_key else None
print("OPENAI_API_KEY set:", bool(api_key), f"({masked})" if masked else "(None)")



Loaded environment from .env
OPENAI_API_KEY set: True (sk-s***mWMA)


In [5]:
domain_prompt = """
### 1. ROLE AND GOAL
You are an Expert-level Software Engineer and Test Set Generator at 'i-sprint innovations.' Your goal is to create a "golden" evaluation dataset for a new RAG system. This dataset will test the RAG's ability to provide accurate, grounded technical support to developers and deployment engineers working with the i-sprint product suite (like UAS).

### 2. TASK
You will be provided with a technical document chunk. Your task is to generate 3-5 high-quality, complex question-answer pairs based *exclusively* on this text. You must output a valid JSON list.

### 3. GENERATION RULES
- **100% GROUNDED:** The `question` must be answerable *only* with the provided text. The `ground_truth_answer` must be a concise, factual summary of the answer. The `source_context` must be the *exact quote(s)* from the text that support the answer.
- **PERSONA-DRIVEN:** Questions must sound like they are from a technical engineer. Use professional, specific, and concise phrasing.
- **TOPIC FOCUS:** Questions must target key identity and security concepts: authentication flows, identity federation (SAML, OIDC), authorization policies, OATH token implementation, security configurations, deployment steps, or troubleshooting error codes.
- **DIVERSE SCENARIOS:** You must generate questions from at least two of the following `question_type` categories:
    1.  **Deployment/Configuration:** Questions about setup, high-availability, or setting parameters.
    2.  **Troubleshooting/Error Handling:** Questions about resolving errors, log analysis, or audit procedures.
    3.  **Integration/Development:** Questions about API endpoints, coding standards, or integrating with protocols like OAuth2.

### 4. OUTPUT FORMAT
Respond ONLY with a valid JSON list. Do not include any text before or after the JSON.

[
  {
    "question_type": "Deployment/Configuration | Troubleshooting/Error Handling | Integration/Development",
    "question": "A specific, professional question based *only* on the document.",
    "ground_truth_answer": "A concise, factual answer derived *only* from the document.",
    "source_context": "The exact quote or passage from the document that contains the answer."
  },
  {
    "question_type": "...",
    "question": "...",
    "ground_truth_answer": "...",
    "source_context": "..."
  }
]

"""


In [6]:
# Import Persona class and create persona objects
from ragas.testset.persona import Persona

personas = [
    Persona(
        name="DevOps Engineer",
        role_description="Troubleshoots production deployments, version upgrades, and compatibility issues. Asks scenario-based questions about migration paths and configuration conflicts."
    ),
    Persona(
        name="Security Architect", 
        role_description="Evaluates authentication protocols, authorization flows, and compliance requirements. Asks about OAuth/SAML implementations and security implications."
    )
]



In [7]:
# RAGAS setup and batch generation for all PDFs
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.testset import TestsetGenerator
from ragas.testset.synthesizers import (
    SingleHopSpecificQuerySynthesizer,
    MultiHopAbstractQuerySynthesizer,
    MultiHopSpecificQuerySynthesizer
)
import json
import pandas as pd
import time

# Setup generator once
# We use GPT-4o for high quality generation
generator_llm = LangchainLLMWrapper(ChatOpenAI(model='gpt-4o', temperature=0.1))
generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model='text-embedding-3-small'))

generator = TestsetGenerator(
    llm=generator_llm,
    embedding_model=generator_embeddings,
    persona_list=personas
)

# Define the distribution of query types
query_distribution = [
    (SingleHopSpecificQuerySynthesizer(llm=generator_llm), 0.2),
    (MultiHopSpecificQuerySynthesizer(llm=generator_llm), 0.6), 
    (MultiHopAbstractQuerySynthesizer(llm=generator_llm), 0.2)
]

# Generate questions for each PDF
generated_datasets = []

# Configuration for batching
BATCH_SIZE = 10  # Save progress every 10 questions

for pdf_data in all_pdf_data:
    pdf_name = pdf_data['pdf_name']
    target_size = pdf_data['testset_size']
    
    print(f"\n  Processing: {pdf_name}")
    print(f"   Target: {target_size} questions")
    
    # Setup output paths for checkpointing
    output_subdir = OUTPUT_DIR / pdf_name
    output_subdir.mkdir(parents=True, exist_ok=True)
    jsonl_path = output_subdir / "test.jsonl"
    csv_path = output_subdir / "test.csv"
    
    # 1. Load existing progress (Resume capability)
    existing_records = []
    if jsonl_path.exists():
        try:
            with open(jsonl_path, "r", encoding="utf-8") as f:
                for line in f:
                    if line.strip():
                        existing_records.append(json.loads(line))
            print(f"   Found existing {len(existing_records)} questions. Resuming...")
        except Exception as e:
            print(f"   Error reading existing file: {e}. Starting over.")
            existing_records = []
    
    # 2. Check if we are already done
    if len(existing_records) >= target_size:
        print(f"   Skipping - Target size reached ({len(existing_records)}/{target_size})")
        # Reconstruct DataFrame for the 'generated_datasets' list so subsequent cells work
        final_df = pd.DataFrame(existing_records)
        generated_datasets.append({
            'pdf_name': pdf_name,
            'df': final_df,
            'testset_size': target_size
        })
        continue

    # 3. Batch Generation Loop
    while len(existing_records) < target_size:
        current_count = len(existing_records)
        remaining = target_size - current_count
        # Generate only what is needed, but in chunks of BATCH_SIZE
        batch_size = min(BATCH_SIZE, remaining)
        
        print(f"   Generating batch of {batch_size} questions ({current_count}/{target_size})...")
        
        try:
            # Generate the batch
            dataset = generator.generate_with_langchain_docs(
                pdf_data['chunks'],
                testset_size=batch_size,
                query_distribution=query_distribution
            )
            
            batch_df = dataset.to_pandas()
            
            # Convert to standard format
            new_records = []
            for _, row in batch_df.iterrows():
                q, gt, ctx = infer_columns(row)
                new_records.append({
                    "query": q,
                    "ground_truth": gt,
                    "contexts": ctx,
                })
            
            # Add to our accumulated list
            existing_records.extend(new_records)
            
            # 4. Save Checkpoint (Overwrite file with updated list)
            with open(jsonl_path, "w", encoding="utf-8") as f:
                for r in existing_records:
                    f.write(json.dumps(r, ensure_ascii=False) + "\n")
            
            # Also update CSV
            pd.DataFrame({
                "query": [r["query"] for r in existing_records],
                "ground_truth": [r["ground_truth"] for r in existing_records],
                "contexts_joined": ["\n\n".join(r["contexts"]) for r in existing_records]
            }).to_csv(csv_path, index=False, encoding="utf-8")
            
            print(f"   Saved checkpoint: {len(existing_records)} questions total.")
            
        except Exception as e:
            print(f"    Error in batch generation: {e}")
            print("    Retrying in 5 seconds...")
            time.sleep(5)
            # We continue the loop to retry
            continue

    # Add completed dataset to memory
    final_df = pd.DataFrame(existing_records)
    generated_datasets.append({
        'pdf_name': pdf_name,
        'df': final_df,
        'testset_size': target_size
    })

print(f"\n Successfully generated datasets for {len(generated_datasets)} PDFs")

C:\Users\junhongs\AppData\Local\Temp\ipykernel_49448\1349355949.py:17: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  generator_llm = LangchainLLMWrapper(ChatOpenAI(model='gpt-4o', temperature=0.1))
C:\Users\junhongs\AppData\Local\Temp\ipykernel_49448\1349355949.py:18: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model='text-embedding-3-small'))
C:\Users\junhongs\AppData\Local\Temp\ipykernel_49448\1349355949.py:18: DeprecationWarning: LangchainEmbedding


  Processing: USO-Administration-Guide-6.0.1-GA (AI)
   Target: 100 questions
   Generating batch of 10 questions (0/100)...


Applying CustomNodeFilter:   0%|          | 0/558 [00:00<?, ?it/s]Node e7b2c0a3-9dfe-4240-90ca-e00f47ddafd4 does not have a summary. Skipping filtering.
Node 498981bf-8fce-4b53-a298-66bc2bc169af does not have a summary. Skipping filtering.
Node 0523bb9a-f381-48e2-b224-ea588a00c849 does not have a summary. Skipping filtering.
Node c86222b1-04a9-4e46-8d4a-e933a58aa364 does not have a summary. Skipping filtering.
Node 2290c22b-b9c2-4a9d-96dc-849db6a3bc38 does not have a summary. Skipping filtering.
Node 9b53d417-4c60-4915-a6b2-1a0cfb244cd8 does not have a summary. Skipping filtering.
Node 2a3ba1e2-ee05-44c8-8875-88c17b09f537 does not have a summary. Skipping filtering.
Applying CustomNodeFilter:   0%|          | 0/558 [00:00<?, ?it/s]Node e7b2c0a3-9dfe-4240-90ca-e00f47ddafd4 does not have a summary. Skipping filtering.
Node 498981bf-8fce-4b53-a298-66bc2bc169af does not have a summary. Skipping filtering.
Node 0523bb9a-f381-48e2-b224-ea588a00c849 does not have a summary. Skipping filtering

   Saved checkpoint: 10 questions total.
   Generating batch of 10 questions (10/100)...


Applying CustomNodeFilter:   0%|          | 0/558 [00:00<?, ?it/s]Node cd733d60-8c63-417c-8f00-1f4d6383c3b1 does not have a summary. Skipping filtering.
Node 0ca2c90c-b285-4475-b8a0-14726e53e80b does not have a summary. Skipping filtering.
Node e4911211-15ac-4a86-9ae5-47f0f7f19594 does not have a summary. Skipping filtering.
Node c5daa05e-ac35-436a-8a00-3ee8d4571ddb does not have a summary. Skipping filtering.
Node e22df2ef-9373-4d7a-989e-7bc032b70295 does not have a summary. Skipping filtering.
Node 9cd50f72-fe4c-4ded-bead-6c9844e29f58 does not have a summary. Skipping filtering.
Node 7569c78d-e935-49be-a933-e745454f3a5b does not have a summary. Skipping filtering.
Node 02b7c5f2-1bec-4d8f-a1ca-c4af863082e2 does not have a summary. Skipping filtering.
Node 2459d7fe-b5ad-43a8-a46d-7f31a582b755 does not have a summary. Skipping filtering.
Node 53ae924e-0aa5-4137-908d-5a2c53e0bcae does not have a summary. Skipping filtering.
Node a7fc015a-2e87-4f1a-aa47-fe53fced4341 does not have a summar

   Saved checkpoint: 20 questions total.
   Generating batch of 10 questions (20/100)...


Applying CustomNodeFilter:   0%|          | 0/558 [00:00<?, ?it/s]Node ef6bf70e-ad54-473d-b555-46a1455a6f69 does not have a summary. Skipping filtering.
Node e31ef20f-df6b-405c-bc91-b91eaeec85d8 does not have a summary. Skipping filtering.
Node 7f85bf1b-dc55-4a89-8b7f-333be3c90de5 does not have a summary. Skipping filtering.
Node d75fcc90-c62e-40ab-8a13-66fbf8f08ab3 does not have a summary. Skipping filtering.
Node 052d18f9-0d3a-4290-8fe1-de182d5212f2 does not have a summary. Skipping filtering.
Node bf6ed7ed-2714-423d-8692-e54df7acd8d5 does not have a summary. Skipping filtering.
Node b30074bd-6240-4604-bc1e-b4a5ac0936be does not have a summary. Skipping filtering.
Node bd8b4594-5d90-47d2-9ff8-06ead13c846e does not have a summary. Skipping filtering.
Node 3001c01e-1c97-44ad-89dc-012c17d685f8 does not have a summary. Skipping filtering.
Node b4ea56ca-aca4-48e7-8a67-237fa953a0ac does not have a summary. Skipping filtering.
Node 84b3b1c3-fd31-4b45-8b0c-f9ab942f9e00 does not have a summar

KeyboardInterrupt: 

In [ ]:
# Save all datasets to individual JSONL and CSV files
import json
import pandas as pd

for data in generated_datasets:
    pdf_name = data['pdf_name']
    df = data['df']
    
    # Create output directory for this PDF
    output_subdir = OUTPUT_DIR / pdf_name
    output_subdir.mkdir(parents=True, exist_ok=True)
    
    jsonl_path = output_subdir / "100qa.jsonl"
    csv_path = output_subdir / "100qa.csv"
    
    # Convert DataFrame rows to standardized format
    records = []
    for _, row in df.iterrows():
        q, gt, ctx = infer_columns(row)
        records.append({
            "query": q,
            "ground_truth": gt,
            "contexts": ctx,
        })
    
    # Write JSONL
    with open(jsonl_path, "w", encoding="utf-8") as f:
        for r in records:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    
    # Write CSV
    pd.DataFrame({
        "query": [r["query"] for r in records],
        "ground_truth": [r["ground_truth"] for r in records],
        "contexts_joined": ["\n\n".join(r["contexts"]) for r in records]
    }).to_csv(csv_path, index=False, encoding="utf-8")
    
    print(f" {pdf_name}")
    print(f"   └─ {output_subdir.relative_to(ROOT)}")
    print(f"      ├─ {jsonl_path.name} ({len(records)} questions)")
    print(f"      └─ {csv_path.name}")

print(f"\n🎉 All datasets saved to: {OUTPUT_DIR.relative_to(ROOT)}")